# Day 3 — 추론 서버 (FastAPI + 배칭)

Day 1에서는 노트북 안에서 직접 `tokenizer` + `model`을 다루고 `pipeline()`으로 텍스트를 생성했습니다. 오늘은 그 모델을 **네트워크 너머의 다른 프로세스/다른 컴퓨터에서도 쓸 수 있는 서버**로 감쌉니다.

오늘 다룰 것:

1. **FastAPI 추론 서버** — `/generate` 엔드포인트로 프롬프트를 받아 텍스트를 반환
2. **배칭(batching) 로직** — 짧은 시간 동안 들어온 여러 요청을 모아 `model.generate()`에 한 번에 넘기는 asyncio 큐 패턴
3. **처리량(throughput) 비교** — 배칭 있음 vs 없음, 동시 요청을 던져서 실측

### 왜 배칭이 필요한가?

GPU는 요청 1개를 처리하든 4개를 처리하든 커널 실행 오버헤드가 비슷합니다. 즉 배치 크기를 늘리면 "요청당 비용"이 줄어듭니다. 반대로 요청이 올 때마다 즉시 1개씩 처리하면(=배칭 없음) GPU가 놀리는 시간(idle)과 커널 launch 오버헤드가 매 요청마다 반복됩니다.

실무 표준 패턴(vLLM 등 프로덕션 서빙 엔진도 기본적으로 이 아이디어의 정교한 버전을 씁니다)은 다음과 같습니다:

- 요청이 들어오면 즉시 처리하지 않고 **큐(queue)**에 넣는다
- 백그라운드 워커가 **짧은 시간 창(time window)** 동안 큐에서 요청을 모은다 (최대 배치 크기에 도달하거나 시간 창이 끝나면 마감)
- 모은 요청들을 **하나의 배치**로 묶어 `model.generate()`를 한 번 호출한다
- 결과를 각 요청에 나눠 돌려준다 (`asyncio.Future`로 개별 응답을 기다리게 함)

8GB 통합 메모리라는 제약 때문에 오늘은 배치 크기를 **2~4** 수준으로 작게 잡습니다. 데스크탑 GPU 서버라면 배치 32~64도 흔하지만, Jetson에서는 배치가 커질수록 KV 캐시 메모리도 비례해서 늘어나 OOM 위험이 커집니다.

In [ ]:
import os

# 주의: 이 노트북의 커널은 Jetson에서 실행되지만, 이 .ipynb 파일 자체는
# codeql-host/작업 PC의 git 저장소에 있습니다. 즉 상대경로 "../serving"은
# 저장소의 serving/ 폴더가 아니라 Jetson 쪽 홈 디렉토리 기준으로 생성됩니다.
# 그래서 여기서는 Jetson 쪽 절대경로를 명시적으로 사용하고, 실습이 끝나면
# 완성된 server.py를 저장소의 serving/ 폴더로 직접 복사해 버전관리에 반영하세요.
SERVER_DIR = os.path.expanduser("~/jetson_mlops_serving")
os.makedirs(SERVER_DIR, exist_ok=True)
print(SERVER_DIR)

## 1. 서버 코드 작성 (`serving/server.py`)

서버는 노트북이 아니라 **별도 파일**로 작성합니다. 실제 운영에서는 노트북이 아니라 이런 스크립트를 `uvicorn`으로 띄우기 때문입니다.

핵심 설계:

- 모델은 프로세스 시작 시 **딱 1번**만 로드합니다 (요청마다 로드하면 8GB 메모리로는 감당 불가).
- `/generate` : 큐에 넣고 배치 워커가 처리 (배칭 적용)
- `/generate_nobatch` : 요청이 오는 즉시 단건으로 처리 (배칭 없음, 비교용 베이스라인)
- `/health` : 헬스체크 (서버가 떴는지, 모델 로딩이 끝났는지 확인용)

**디코더 전용 모델을 배치로 `generate()`할 때 주의할 점**: 프롬프트마다 길이가 다르므로 패딩(padding)이 필요한데, causal LM은 반드시 **left padding**을 써야 합니다 (오른쪽에 패딩을 넣으면 다음 토큰 예측 위치가 어긋납니다). `tokenizer.padding_side = "left"`로 설정합니다.

**GPU 연산은 블로킹(blocking)**이라는 점도 중요합니다. `model.generate()`를 async 함수 안에서 그냥 호출하면 이벤트 루프 전체가 멈춰서 다른 요청을 받을 수도, 큐를 처리할 수도 없습니다. 그래서 실제 생성 함수는 `asyncio.to_thread()`로 별도 스레드에서 실행합니다.

In [ ]:
SERVER_CODE = """\"\"\"
Day 3 - 추론 서버 (FastAPI + 배칭)
Jetson Orin Nano (JetPack 7.2, sm_87, 8GB 통합 메모리) 대상.

실행:
    cd ~/jetson_mlops_serving && python -m uvicorn server:app --host 127.0.0.1 --port 8000

설계 포인트
- Qwen2.5-0.5B-Instruct 모델을 프로세스 시작 시 1회만 로드 (메모리 절약)
- /generate         : asyncio.Queue 기반 마이크로배칭 적용 엔드포인트
- /generate_nobatch : 배칭 없이 요청을 받는 즉시 개별 처리 (비교용 베이스라인)
- /health           : 헬스체크
- 8GB 통합 메모리 제약을 감안해 MAX_BATCH_SIZE를 4로 작게 제한
\"\"\"

import asyncio
import time
from dataclasses import dataclass
from typing import Optional

import torch
from fastapi import FastAPI
from pydantic import BaseModel
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
MAX_BATCH_SIZE = 4            # 8GB 통합 메모리 제약상 작게 유지 (2~4 권장)
BATCH_WINDOW_SEC = 0.15       # 요청을 모으는 시간 창 (150ms)
MAX_NEW_TOKENS_DEFAULT = 32

device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"[server] loading {MODEL_NAME} on {device} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float16).to(device)
model.eval()
print("[server] model loaded.")

app = FastAPI(title="Jetson MLOps Lab - Day3 Serving")


class GenerateRequest(BaseModel):
    prompt: str
    max_new_tokens: int = MAX_NEW_TOKENS_DEFAULT


class GenerateResponse(BaseModel):
    text: str
    batch_size: int
    latency_sec: float


@dataclass
class PendingRequest:
    prompt: str
    max_new_tokens: int
    future: "asyncio.Future"


request_queue: Optional[asyncio.Queue] = None


@app.on_event("startup")
async def startup():
    global request_queue
    request_queue = asyncio.Queue()
    asyncio.create_task(batch_worker())


def generate_batch(prompts, max_new_tokens):
    messages_batch = [[{"role": "user", "content": p}] for p in prompts]
    chat_prompts = [
        tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
        for m in messages_batch
    ]
    inputs = tokenizer(chat_prompts, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    gen_only = output_ids[:, inputs["input_ids"].shape[1]:]
    return tokenizer.batch_decode(gen_only, skip_special_tokens=True)


async def run_batch(batch):
    start = time.monotonic()
    prompts = [item.prompt for item in batch]
    max_new = max(item.max_new_tokens for item in batch)
    texts = await asyncio.to_thread(generate_batch, prompts, max_new)
    elapsed = time.monotonic() - start
    for item, text in zip(batch, texts):
        if not item.future.done():
            item.future.set_result((text, len(batch), elapsed))


async def batch_worker():
    while True:
        first = await request_queue.get()
        batch = [first]
        deadline = time.monotonic() + BATCH_WINDOW_SEC
        while len(batch) < MAX_BATCH_SIZE:
            remaining = deadline - time.monotonic()
            if remaining <= 0:
                break
            try:
                nxt = await asyncio.wait_for(request_queue.get(), timeout=remaining)
                batch.append(nxt)
            except asyncio.TimeoutError:
                break
        await run_batch(batch)


@app.get("/health")
async def health():
    return {"status": "ok", "device": device}


@app.post("/generate", response_model=GenerateResponse)
async def generate(req: GenerateRequest):
    loop = asyncio.get_running_loop()
    fut = loop.create_future()
    pending = PendingRequest(prompt=req.prompt, max_new_tokens=req.max_new_tokens, future=fut)
    await request_queue.put(pending)
    text, batch_size, elapsed = await fut
    return GenerateResponse(text=text, batch_size=batch_size, latency_sec=elapsed)


@app.post("/generate_nobatch", response_model=GenerateResponse)
async def generate_nobatch(req: GenerateRequest):
    start = time.monotonic()
    texts = await asyncio.to_thread(generate_batch, [req.prompt], req.max_new_tokens)
    elapsed = time.monotonic() - start
    return GenerateResponse(text=texts[0], batch_size=1, latency_sec=elapsed)
"""

with open(os.path.join(SERVER_DIR, "server.py"), "w", encoding="utf-8") as f:
    f.write(SERVER_CODE)

print("wrote", os.path.join(SERVER_DIR, "server.py"))

**코드 설명:**

- `/generate`는 큐에 `PendingRequest`(프롬프트 + 빈 `Future`)를 넣고 `await fut`로 자기 결과가 채워질 때까지 기다립니다. 이 동안에도 이벤트 루프는 다른 요청을 계속 받을 수 있습니다 (비동기니까요).
- `batch_worker()`는 무한 루프를 돌며: 큐에서 요청 1개를 꺼내고, 그 순간부터 `BATCH_WINDOW_SEC`(150ms) 동안 추가 요청을 최대 `MAX_BATCH_SIZE`(4)개까지 더 모읍니다. 시간이 다 되거나 배치가 꽉 차면 마감하고 `run_batch()`로 실제 생성을 실행합니다.
- `run_batch()`는 모인 요청들의 프롬프트를 한 번에 `model.generate()`에 넘기고, 결과를 각 요청의 `Future`에 나눠 채워줍니다. 이게 배칭의 핵심입니다: **N개 요청 → GPU 호출 1번**.
- `/generate_nobatch`는 큐를 거치지 않고 요청이 오자마자 바로 `generate_batch([prompt], ...)`를 호출합니다 (배치 크기 1). 동시에 여러 요청이 오면 스레드풀에서 각각 별도로 GPU 호출을 하게 되어, 매 요청마다 커널 launch 오버헤드가 반복됩니다.

## 2. 서버를 백그라운드 프로세스로 실행

실제 배포 환경에서는 `uvicorn`을 별도 터미널/프로세스로 띄웁니다. 노트북 안에서는 `subprocess.Popen`으로 uvicorn을 백그라운드에 띄운 뒤, `/health` 엔드포인트가 응답할 때까지 폴링하여 모델 로딩이 끝나기를 기다립니다.

(모델 로딩에는 첫 실행 시 다운로드 포함 수십 초~1~2분이 걸릴 수 있습니다.)

In [ ]:
import subprocess
import sys
import time
import requests

LOG_PATH = os.path.join(SERVER_DIR, "server.log")

log_file = open(LOG_PATH, "w", encoding="utf-8")
server_proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "server:app", "--host", "127.0.0.1", "--port", "8000"],
    cwd=SERVER_DIR,
    stdout=log_file,
    stderr=subprocess.STDOUT,
)
print("uvicorn PID:", server_proc.pid)

# /health가 200을 반환할 때까지 폴링 (모델 로딩 완료 대기)
HEALTH_URL = "http://127.0.0.1:8000/health"
max_wait_sec = 180
waited = 0
ready = False
while waited < max_wait_sec:
    try:
        r = requests.get(HEALTH_URL, timeout=2)
        if r.status_code == 200:
            ready = True
            break
    except requests.exceptions.ConnectionError:
        pass
    time.sleep(2)
    waited += 2

if ready:
    print(f"서버 준비 완료 ({waited}초 소요):", r.json())
else:
    print("서버가 응답하지 않습니다. 로그를 확인하세요:", LOG_PATH)
    with open(LOG_PATH, encoding="utf-8") as f:
        print(f.read()[-2000:])

## 3. 동작 확인: 요청 1개 보내보기

배칭/비교 테스트 전에, 두 엔드포인트가 정상 응답하는지 먼저 단건으로 확인합니다.

In [ ]:
resp = requests.post(
    "http://127.0.0.1:8000/generate",
    json={"prompt": "Jetson Orin Nano의 장점을 한 문장으로 말해줘.", "max_new_tokens": 32},
    timeout=60,
)
print(resp.status_code)
print(resp.json())

## 4. 처리량(throughput) 비교: 배칭 없음 vs 있음

이제 여러 요청을 **동시에** 쏴서 두 엔드포인트의 처리량을 비교합니다. 방법: `httpx.AsyncClient`로 N개의 요청을 `asyncio.gather`를 통해 동시에 보내고, 전체가 끝날 때까지 걸린 시간으로 처리량(`요청수/초`)을 계산합니다.

주의: 8GB 메모리 제약상 동시 요청 수(`N_REQUESTS`)를 너무 크게 잡지 않습니다 (`/generate_nobatch`는 배칭 없이 각 요청이 개별 GPU 호출을 하므로, 동시 요청이 너무 많으면 스레드풀에서 몰려서 실행되며 메모리 압박이 커질 수 있습니다). 여기서는 8개로 제한합니다.

In [ ]:
import httpx

N_REQUESTS = 8
MAX_NEW_TOKENS = 32
PROMPTS = [
    f"숫자 {i}에 대해 한 문장으로 흥미로운 사실을 알려줘." for i in range(N_REQUESTS)
]


async def run_throughput_test(endpoint: str, prompts, max_new_tokens=MAX_NEW_TOKENS):
    url = f"http://127.0.0.1:8000/{endpoint}"
    async with httpx.AsyncClient(timeout=120) as client:
        start = time.monotonic()
        tasks = [
            client.post(url, json={"prompt": p, "max_new_tokens": max_new_tokens})
            for p in prompts
        ]
        responses = await asyncio.gather(*tasks)
        elapsed = time.monotonic() - start

    results = [r.json() for r in responses]
    throughput = len(prompts) / elapsed
    return {
        "endpoint": endpoint,
        "n_requests": len(prompts),
        "elapsed_sec": elapsed,
        "throughput_req_per_sec": throughput,
        "batch_sizes_seen": [r["batch_size"] for r in results],
    }


import asyncio

result_nobatch = await run_throughput_test("generate_nobatch", PROMPTS)
result_nobatch

In [ ]:
result_batch = await run_throughput_test("generate", PROMPTS)
result_batch

## 5. 결과 비교 및 해석

In [ ]:
speedup = result_batch["throughput_req_per_sec"] / result_nobatch["throughput_req_per_sec"]

print(f"[배칭 없음]  {result_nobatch['n_requests']}개 요청, {result_nobatch['elapsed_sec']:.2f}초, "
      f"{result_nobatch['throughput_req_per_sec']:.2f} req/s, batch_size 분포={result_nobatch['batch_sizes_seen']}")
print(f"[배칭 있음]  {result_batch['n_requests']}개 요청, {result_batch['elapsed_sec']:.2f}초, "
      f"{result_batch['throughput_req_per_sec']:.2f} req/s, batch_size 분포={result_batch['batch_sizes_seen']}")
print(f"\n처리량 향상 배수: {speedup:.2f}x")

**실측 결과** (이 하드웨어, N_REQUESTS=8, max_new_tokens=32 기준):

```
[배칭 없음]  0.32 req/s, batch_size 분포=[1, 1, 1, 1, 1, 1, 1, 1]
[배칭 있음]  1.32 req/s, batch_size 분포=[4, 4, 4, 4, 4, 4, 4, 4]
처리량 향상 배수: 4.09x
```

**관찰 포인트:**

- `배칭 있음`에서 8개 요청이 정확히 4개씩 두 배치로 묶여 처리됐습니다 (`MAX_BATCH_SIZE=4`에 도달할 때까지 모은 뒤 마감).
- `배칭 없음`은 각 요청이 batch_size 1로 개별 처리되며, `asyncio.to_thread`로 스레드풀에 넘겨도 GPU 자체는 한 번에 하나의 커널만 실행하므로 사실상 순차 처리와 비슷하게 느려집니다.
- 실측 배수가 4배 이상으로, 처음 예상했던 "1.3~2배"보다 훨씬 큽니다 — 이 모델(0.5B)+짧은 prompt 조합에서는 GPU 커널 launch/모델 forward의 고정 오버헤드 비중이 커서, 배치로 묶어 오버헤드를 나눠 갚는 효과가 예상보다 크게 나타난 것으로 보입니다. 실제 수치는 모델 크기·prompt 길이·동시 요청 패턴에 따라 달라지니 직접 재현해보고 비교해보세요.
- `BATCH_WINDOW_SEC`(150ms)를 늘리면 배치가 더 꽉 차서(처리량↑) 개별 요청의 지연시간(latency)은 늘어나는 트레이드오프가 생깁니다. 직접 이 값을 50ms/300ms 등으로 바꿔가며 `server.log`와 결과를 비교해보세요.
- 8GB 통합 메모리 환경에서는 `MAX_BATCH_SIZE`를 8 이상으로 올리면 (특히 `max_new_tokens`가 크면) KV 캐시 메모리가 늘어나 OOM 위험이 커집니다.

## 6. 서버 종료 (정리)

노트북을 계속 쓰기 전에, 백그라운드로 띄운 uvicorn 프로세스를 반드시 종료해서 GPU 메모리를 반납합니다.

In [ ]:
server_proc.terminate()
try:
    server_proc.wait(timeout=10)
except subprocess.TimeoutExpired:
    server_proc.kill()
    server_proc.wait()
log_file.close()
print("서버 종료 완료.")

## 정리 (Day 3 요약)

- FastAPI로 텍스트 생성 엔드포인트를 만들고, `subprocess`로 백그라운드에 띄워 노트북에서 실제 HTTP 요청으로 테스트하는 워크플로우를 익혔습니다.
- `asyncio.Queue` + 시간 창(time window) 기반의 마이크로배칭 패턴을 구현했습니다: 요청이 오면 즉시 처리하지 않고 짧은 시간 동안 모아서 배치로 `model.generate()`에 넘기는 방식입니다. 이는 vLLM, TGI 등 프로덕션 서빙 엔진이 쓰는 **continuous/dynamic batching**의 단순화된 버전입니다.
- 배칭 유무에 따른 처리량을 실측 비교해, 배치 크기를 늘릴수록(단, 메모리 한도 내에서) GPU 활용률과 처리량이 개선됨을 확인했습니다.
- 8GB 통합 메모리라는 제약 때문에 배치 크기를 2~4로 작게 유지했고, 이 트레이드오프(처리량 vs 지연시간 vs 메모리)를 직접 조절해볼 수 있는 파라미터(`MAX_BATCH_SIZE`, `BATCH_WINDOW_SEC`)를 코드에 남겨두었습니다.

**다음 단계로 생각해볼 것**: 이 서버는 요청이 순서대로 큐에 쌓이는 단순한 형태입니다. 실제로는 (1) 요청마다 다른 `max_new_tokens`로 인해 배치 안에서 짧은 생성이 긴 생성을 기다려야 하는 문제(→ continuous batching으로 해결), (2) 여러 클라이언트의 동시 접속과 백프레셔(backpressure) 처리, (3) 스트리밍 응답(SSE) 등이 추가로 다뤄질 만한 주제입니다.

**저장소에 반영하기**: 완성된 서버 코드는 Jetson 쪽 `~/jetson_mlops_serving/server.py`에 있습니다. 버전관리를 위해 이 파일 내용을 저장소의 `serving/server.py`로 복사해 커밋하세요 (커널과 파일이 서로 다른 머신에 있다는 이 프로젝트의 구조적 특징 때문에 자동으로 동기화되지 않습니다 - CLAUDE.md 참고).